In [1]:
from collections import defaultdict
import numpy as np
import gymnasium as gym


def fixed_policy(state):
    """Fixed deterministic policy: Hit if sum < 20, else Stick."""
    player_sum, _, _ = state
    return 1 if player_sum < 20 else 0


def generate_episode(env, policy):
    """Plays a complete game of blackjack using the fixed policy."""
    episode = []
    state, info = env.reset()
    done = False

    while not done:
        action = policy(state)

        next_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated

        episode.append((state, action, reward))
        state = next_state

    return episode


def run_mc_prediction(num_episodes=500000):
    """Runs First-Visit Monte Carlo Prediction to evaluate V(s)."""
    env = gym.make('Blackjack-v1', sab=True)

    V = defaultdict(float)
    N = defaultdict(int)

    print(f"Running MC Prediction for {num_episodes:,} episodes...")

    for _ in range(num_episodes):
        episode = generate_episode(env, fixed_policy)

        # Extract the reward from the last step
        G = episode[-1][2]

        visited_states = set()

        for state, _, _ in episode:
            if state not in visited_states:
                visited_states.add(state)
                N[state] += 1
                V[state] += (1 / N[state]) * (G - V[state])

    env.close()
    return V


if __name__ == "__main__":
    V_table = run_mc_prediction(500000)

    print("\nPrediction Complete! Sample state values:")
    print("-" * 50)

    for state in [(21, 10, False), (14, 7, False), (12, 5, False)]:
        print(f"State {state}: V(s) = {V_table.get(state, 0.0):+.4f}")

Running MC Prediction for 500,000 episodes...

Prediction Complete! Sample state values:
--------------------------------------------------
State (21, 10, False): V(s) = +0.8876
State (14, 7, False): V(s) = -0.5931
State (12, 5, False): V(s) = -0.5144
